In [65]:
import scipy.io as sio
import numpy as np
import os
import glob
from matplotlib import pyplot as plt
import re

In [32]:
data_path = "/path/to/mae/data/fixed_data"

In [33]:
target_files = glob.glob(os.path.join(data_path, "*.mat"))

In [34]:
len(target_files)

72

In [35]:
# Group files by battery name and state (charge/discharge)
file_groups = {}

for file_path in target_files:
    # Extract filename from path
    filename = os.path.basename(file_path)
    
    # Extract battery name and state
    # Pattern: {battery_name}_{state}_wo_{number}.mat
    parts = filename.split('_wo_')[0]
    battery_name, state = parts.split('_', 1)
    
    # Create key for grouping
    group_key = (battery_name, state)
    
    # Add file to group
    if group_key not in file_groups:
        file_groups[group_key] = []
    file_groups[group_key].append(file_path)

# Display groups and counts
for (battery, state), files in file_groups.items():
    print(f"{battery} - {state}: {len(files)} files")
    # Uncomment below to see all files in each group
    files = sorted(files)
    for f in files:
        print(f"  - {os.path.basename(f)}")

NMC#4 - discharge: 5 files
  - NMC#4_discharge_wo_12.mat
  - NMC#4_discharge_wo_14.mat
  - NMC#4_discharge_wo_15.mat
  - NMC#4_discharge_wo_16.mat
  - NMC#4_discharge_wo_17.mat
NMC#2 - discharge: 8 files
  - NMC#2_discharge_wo_11.mat
  - NMC#2_discharge_wo_12.mat
  - NMC#2_discharge_wo_13.mat
  - NMC#2_discharge_wo_14.mat
  - NMC#2_discharge_wo_15.mat
  - NMC#2_discharge_wo_16.mat
  - NMC#2_discharge_wo_17.mat
  - NMC#2_discharge_wo_18.mat
LCO#4 - discharge: 7 files
  - LCO#4_discharge_wo_10.mat
  - LCO#4_discharge_wo_11.mat
  - LCO#4_discharge_wo_12.mat
  - LCO#4_discharge_wo_14.mat
  - LCO#4_discharge_wo_15.mat
  - LCO#4_discharge_wo_16.mat
  - LCO#4_discharge_wo_17.mat
NMC#3 - discharge: 8 files
  - NMC#3_discharge_wo_10.mat
  - NMC#3_discharge_wo_12.mat
  - NMC#3_discharge_wo_13.mat
  - NMC#3_discharge_wo_14.mat
  - NMC#3_discharge_wo_15.mat
  - NMC#3_discharge_wo_16.mat
  - NMC#3_discharge_wo_17.mat
  - NMC#3_discharge_wo_18.mat
NMC#2 - charge: 8 files
  - NMC#2_charge_wo_0.mat
  

In [8]:
def draw_total_data(show_data, files, result_path, filename):
    plt.figure(figsize=(16, 2 * len(files)))
    # plt.imshow(show_data, aspect='auto', cmap='seismic', vmin=-0.5, vmax=0.5)
    plt.imshow(show_data, aspect='auto', cmap='seismic')
    plt.colorbar()
    plt.axis('off')  # Turn axis on to show filenames
    plt.savefig(os.path.join(result_path, f'{filename}.png'), bbox_inches='tight', dpi=300)
    plt.close()

In [37]:
save_path = "../data/mixup_data"
os.makedirs(save_path, exist_ok=True)
origin_save_path = "../data/mixup_data/original_data"
os.makedirs(origin_save_path, exist_ok=True)
mixup_save_path = "../data/mixup_data/new_mixup_data"
os.makedirs(mixup_save_path, exist_ok=True)
def save_data(save_path, filename, tf):
    if os.path.exists(os.path.join(save_path, filename)):
        return
    else:
        sio.savemat(os.path.join(save_path, filename), tf)

weight = [
    (0.5, 0.5), 
    (0.3, 0.7),
    (0.7, 0.3), 
    (0.2, 0.8), 
    (0.8, 0.2), 
    (0.1, 0.9), 
    (0.9, 0.1)
]

for (battery, state), files in file_groups.items():
    files = sorted(files)
    for idx in range(len(files)-1):
        
        stage1 = int(re.search(r'wo_(\d+)', files[idx]).group(1))
        stage2 = int(re.search(r'wo_(\d+)', files[idx+1]).group(1))
        
        tf1 = sio.loadmat(files[idx])
        filename1 = os.path.basename(files[idx])
        save_data(origin_save_path, filename1, tf1)
        data1= tf1['data']
        mah1 = tf1['ah'][0][0]
        
        tf2 = sio.loadmat(files[idx+1])
        filename2 = os.path.basename(files[idx+1])
        save_data(origin_save_path, filename2, tf2)
        data2 = tf2['data']
        mah2 = tf2['ah'][0][0]
        
        for wt1, wt2 in weight:
            new_data = wt1 * data1 + wt2 * data2
            new_mah = wt1 * mah1 + wt2 * mah2
            new_tf = {'data': new_data, 'ah': np.array([[new_mah]])}
            new_filename = f"{battery}_{state}_wo_{round(wt1 * stage1 + wt2 * stage2, 2)}.mat"
            save_data(mixup_save_path, new_filename, new_tf)
        

In [ ]:
# 将所有数据绘制在一张大图上
result_path = "../vision_batter/origin"
os.makedirs(result_path, exist_ok=True)
for (battery, state), files in file_groups.items():
    total_data = []
    files = sorted(files)
    for file_path in files:
        tf = sio.loadmat(file_path)
        # Get all channels from data
        data = tf['data']
        rows = data.shape[0]
        chans = data.shape[-1]
        cat_data = data[:, :, 0]
        for i in range(chans-1):
            cat_data = np.concatenate([cat_data, np.zeros((rows,2)) + 1, data[:, :, i+1]], axis=1)
        
        total_data.append(cat_data)
    
    show_data = total_data[0]
    cols = show_data.shape[1]
    for i in range(len(total_data)-1):
        show_data = np.concatenate([show_data, np.zeros((2,cols)) + 1, total_data[i+1]], axis=0)

    draw_total_data(show_data, files, result_path, f"{battery}_{state}_origin")

In [ ]:
#-------------------修复数据 取6 修nmc 1 dis -------------------
result_path = "../data/fixed_data"
os.makedirs(result_path, exist_ok=True)

for (battery, state), files in file_groups.items():
    if battery == "NMC#1" and state == "discharge":
        for file in files:
            tf = sio.loadmat(file)
            data = tf['data']
            mah = tf['ah'][0][0]
            
            choice_data = data[:, :, : -1]
            choice_data[:,:,-1] = choice_data[:, :, -2] * 1.1
            new_tf = {'data': choice_data, 'ah': np.array([[mah]])}
            new_filename = os.path.basename(file)
            save_data(result_path, new_filename, new_tf)
    else:
        for file in files:
            tf = sio.loadmat(file)
            data = tf['data']
            mah = tf['ah'][0][0]
            choice_data = data[:, :, 1:]
            new_tf = {'data': choice_data, 'ah': np.array([[mah]])}
            new_filename = os.path.basename(file)
            save_data(result_path, new_filename, new_tf)

In [68]:
target_path = "/path/to/mae/data/mixup_data/original_data"
lco_files = glob.glob(os.path.join(target_path, "LCO*.mat"))
nmc_files = glob.glob(os.path.join(target_path, "NMC*.mat"))

In [77]:
target_mixup_path = "/path/to/mae/data/mixup_data/new_mixup_data"
mix_lco_files = glob.glob(os.path.join(target_mixup_path, "LCO*.mat"))
mix_nmc_files = glob.glob(os.path.join(target_mixup_path, "NMC*.mat"))

In [70]:
lco_files = sorted(lco_files)
nmc_files = sorted(nmc_files)

In [79]:
import random
import json

# Set random seed for reproducibility
random.seed(42)

# Randomly select 8 files
selected_lco = random.sample(lco_files, 8)

# Sort the files for better readability
train_lco = sorted([f for f in lco_files if f not in selected_lco])
test_lco = sorted(selected_lco)

In [80]:
import json
# Save splits to JSON
split_data = {
    'train': train_lco + mix_lco_files,
    'test': test_lco
}

with open('../data/lco.json', 'w') as f:
    json.dump(split_data, f, indent=4)

In [81]:
import random

# Set random seed for reproducibility
random.seed(42)

# Randomly select 8 files
selected_nmc = random.sample(nmc_files, 8)

# Sort the files for better readability
train_nmc = sorted([f for f in nmc_files if f not in selected_nmc])
test_nmc = sorted(selected_nmc)



In [82]:
split_data = {
    'train': train_nmc + mix_nmc_files,
    'test': test_nmc
}

with open('../data/nmc.json', 'w') as f:
    json.dump(split_data, f, indent=4)